# Medical Transcription & SOAP Note Generation

This notebook implements a pipeline that:
1. **Transcribes** a medical dictation audio file into text
2. **Generates** a structured SOAP note from the transcript using Generative AI

---

## Architecture Overview

```
┌─────────────────┐     ┌──────────────────────┐     ┌───────────────────┐
│  Audio File      │────▶│  Speech-to-Text       │────▶│  Transcript       │
│  (.mp3/.wav)     │     │  (Whisper / Gemini)   │     │  (verbatim text)  │
└─────────────────┘     └──────────────────────┘     └────────┬──────────┘
                                                               │
                                                               ▼
                        ┌──────────────────────┐     ┌───────────────────┐
                        │  SOAP Note Generator  │────▶│  Structured SOAP  │
                        │  (Gemini LLM + Prompt)│     │  (JSON / Markdown)│
                        └──────────────────────┘     └───────────────────┘
```

### Tool Selection Rationale

| Component | Tool | Why |
|-----------|------|-----|
| Speech-to-Text (Primary) | **faster-whisper** | Local/open-source, HIPAA-friendly — no audio leaves the machine. 4× faster than OpenAI Whisper on CPU via CTranslate2 optimization. |
| Speech-to-Text (Fallback) | **Gemini API** | For hardware-constrained environments without GPU. Handles 13+ minute audio natively via multimodal input. |
| SOAP Structuring | **Gemini API** | Strong medical reasoning, structured JSON output mode, temperature control for deterministic results. |


## 0. Setup & Dependencies

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install google-genai faster-whisper

import os
import json
import logging
from pathlib import Path

# Configure logging for notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s │ %(levelname)-7s │ %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("notebook")

# Project paths
PROJECT_ROOT = Path(".").resolve()
AUDIO_FILE = PROJECT_ROOT / "audio" / "sample_dictation.mp3"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Audio file:   {AUDIO_FILE}")
print(f"Exists:       {AUDIO_FILE.exists()}")
print(f"Size:         {AUDIO_FILE.stat().st_size / 1024 / 1024:.1f} MB" if AUDIO_FILE.exists() else "")

## 1. Audio Transcription (Speech-to-Text)

We provide two transcription backends. The notebook tries the **local Whisper** model first (privacy-centric), and falls back to the **Gemini API** if Whisper is not installed.

### Why faster-whisper?
- **Privacy**: Audio never leaves the local machine — critical for HIPAA compliance
- **Performance**: CTranslate2 engine makes it 4× faster than OpenAI Whisper on CPU
- **Accuracy**: Same Whisper model weights, same accuracy with medical terminology
- **Open Source**: Fully transparent, auditable, no vendor lock-in

In [ ]:
# ──────────────────────────────────────────────────────
# PART A: Audio Transcription
# ──────────────────────────────────────────────────────

transcript = None
transcription_backend = None

# ── Attempt 1: Local faster-whisper ──────────────────
try:
    from faster_whisper import WhisperModel

    MODEL_SIZE = "base"  # Options: tiny, base, small, medium, large-v3
    DEVICE = "cpu"       # Use "cuda" if you have an NVIDIA GPU
    BEAM_SIZE = 5        # Higher = more accurate, slower

    logger.info("Loading faster-whisper model '%s' on %s...", MODEL_SIZE, DEVICE)

    whisper_model = WhisperModel(
        MODEL_SIZE,
        device=DEVICE,
        compute_type="int8",  # Optimized for CPU
    )

    logger.info("Transcribing audio (this may take a few minutes on CPU)...")

    segments, info = whisper_model.transcribe(
        str(AUDIO_FILE),
        beam_size=BEAM_SIZE,
        language="en",
        vad_filter=True,  # Voice Activity Detection to skip silence
        vad_parameters=dict(min_silence_duration_ms=500),
    )

    logger.info("Detected language: %s (probability: %.2f)",
                info.language, info.language_probability)

    # Collect segments into full transcript
    transcript_parts = []
    for segment in segments:
        transcript_parts.append(segment.text.strip())

    transcript = " ".join(transcript_parts)
    transcription_backend = f"faster-whisper ({MODEL_SIZE})"

    logger.info("✓ Local transcription complete (%d characters)", len(transcript))

except ImportError:
    logger.warning("faster-whisper not installed — falling back to Gemini API")
except Exception as e:
    logger.warning("Whisper failed: %s — falling back to Gemini API", e)

# ── Attempt 2: Gemini API fallback ───────────────────
if transcript is None:
    from google import genai
    from google.genai import types

    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
    if not GEMINI_API_KEY:
        raise ValueError(
            "No transcription backend available!\n"
            "Either install faster-whisper or set GEMINI_API_KEY."
        )

    client = genai.Client(api_key=GEMINI_API_KEY)
    TRANSCRIPTION_MODEL = "gemini-3.5-flash-lite"

    logger.info("Uploading audio to Gemini Files API...")
    uploaded_file = client.files.upload(file=str(AUDIO_FILE))
    logger.info("Upload complete: %s", uploaded_file.name)

    logger.info("Requesting transcription from %s...", TRANSCRIPTION_MODEL)

    response = client.models.generate_content(
        model=TRANSCRIPTION_MODEL,
        contents=[
            types.Content(parts=[
                types.Part(
                    file_data=types.FileData(
                        file_uri=uploaded_file.uri,
                        mime_type="audio/mpeg",
                    )
                ),
                types.Part(text=(
                    "Transcribe this medical dictation audio verbatim. "
                    "Output ONLY the transcript text with no labels, headers, or formatting. "
                    "Preserve all medical terminology exactly as spoken. "
                    "Include filler words (um, uh) and natural speech patterns. "
                    "Do not summarize or paraphrase."
                )),
            ])
        ],
    )

    transcript = response.text.strip()
    transcription_backend = f"Gemini API ({TRANSCRIPTION_MODEL})"

    # Clean up
    try:
        client.files.delete(name=uploaded_file.name)
    except Exception:
        pass

    logger.info("✓ Gemini transcription complete (%d characters)", len(transcript))

print(f"\nTranscription backend: {transcription_backend}")
print(f"Transcript length: {len(transcript)} characters")
print(f"\n{'─' * 60}")
print(f"TRANSCRIPT PREVIEW (first 500 chars):")
print(f"{'─' * 60}")
print(transcript[:500] + "...")

In [ ]:
# Save the raw transcript
transcript_path = OUTPUT_DIR / "transcript.txt"
transcript_path.write_text(transcript, encoding="utf-8")
print(f"Transcript saved to: {transcript_path}")

## 2. SOAP Note Generation (Text-to-SOAP)

We use **Google Gemini** with carefully engineered prompts to:
- Parse the unstructured transcript into the four SOAP sections
- Enforce structured JSON output via `response_mime_type`
- Use low temperature (0.2) for deterministic, consistent results

### Prompt Engineering Strategy

The system prompt uses explicit **inclusion/exclusion rules** for each section:
- **S (Subjective)**: Only patient-reported info (symptoms, history, feelings)
- **O (Objective)**: Only doctor-observed/measured data (exam, tests, vitals)
- **A (Assessment)**: Diagnosis based on S + O findings
- **P (Plan)**: Treatment, medications, follow-ups, prognosis

This design minimizes cross-contamination between sections.

In [ ]:
# ──────────────────────────────────────────────────────
# PART B: SOAP Note Generation
# ──────────────────────────────────────────────────────

from google import genai
from google.genai import types

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
SOAP_MODEL = "gemini-3.5-flash-lite"

client = genai.Client(api_key=GEMINI_API_KEY)

# ── System prompt with strict section separation rules ──
SYSTEM_PROMPT = """\
You are a highly skilled medical scribe AI. Your job is to take a raw \
medical dictation transcript and produce a concise, accurate, and \
well-structured SOAP note.

RULES:
1. **Subjective (S):** Only include information the PATIENT reported — \
   symptoms, history, feelings, what makes it better/worse, duration, \
   severity (pain scale), medications tried, and relevant social/family history. \
   Do NOT include anything the doctor observed or measured.

2. **Objective (O):** Only include information the DOCTOR observed, measured, \
   or tested — physical exam findings, vital signs, inspection results, \
   palpation, range of motion tests, and any imaging or lab results. \
   Do NOT include patient-reported symptoms.

3. **Assessment (A):** Provide the diagnosis or differential diagnosis \
   based on the subjective and objective findings. Include the medical \
   term and any common name.

4. **Plan (P):** Include the treatment plan — activity modifications, \
   medications, therapies, follow-up instructions, referrals, and expected \
   prognosis/timeline.

FORMATTING:
- Use concise medical documentation style (abbreviations are OK)
- Use bullet points for multiple items within a section
- Preserve medical terminology exactly
- Include relevant negatives (pertinent negative findings)

OUTPUT:
Return ONLY a valid JSON object with keys: "subjective", "objective", \
"assessment", "plan". Each value is a string containing the section content.
"""

# ── Generate SOAP note ────────────────────────────────
user_prompt = (
    f"Transcript of a medical dictation:\n\n"
    f'\"\"\"\n{transcript}\n\"\"\"\n\n'
    f"Generate a structured SOAP note from this transcript. "
    f"Return ONLY the JSON object — no markdown fences, no explanation."
)

logger.info("Generating SOAP note with %s...", SOAP_MODEL)

response = client.models.generate_content(
    model=SOAP_MODEL,
    contents=user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=0.2,
        response_mime_type="application/json",  # Force JSON output
    ),
)

# Parse the response
soap_data = json.loads(response.text)

logger.info("✓ SOAP note generated successfully")
print("\nSOAP Note (JSON):")
print(json.dumps(soap_data, indent=2))

### 2.1 Cross-Contamination Validation (Bonus)

This check ensures no **Subjective** information (patient-reported) leaked into the **Objective** section (doctor-measured), and vice versa.

In [ ]:
# ──────────────────────────────────────────────────────
# BONUS: Cross-contamination validation
# ──────────────────────────────────────────────────────

def validate_soap_separation(soap: dict) -> list[str]:
    """
    Check that Subjective and Objective sections don't contain
    each other's content types.
    
    Returns a list of warning messages (empty = all clean).
    """
    issues = []
    
    # Patient-reported language that shouldn't be in Objective
    subjective_markers = [
        "patient reports", "patient states", "patient says",
        "patient complains", "patient denies", "patient feels",
        "he reports", "she reports", "he states", "she states",
        "rates the pain", "rates pain", "describes",
        "has been feeling", "has noticed",
    ]
    
    # Doctor-observed language that shouldn't be in Subjective
    objective_markers = [
        "on exam", "on examination", "on inspection",
        "palpation reveals", "upon palpation",
        "range of motion", "rom ", "x-ray", "xray",
        "mri ", "ct scan", "lab results", "labs show",
        "vital signs", "blood pressure", "bp ",
        "temperature", "tenderness on", "swelling noted",
    ]
    
    subj_lower = soap.get("subjective", "").lower()
    obj_lower = soap.get("objective", "").lower()
    
    for marker in objective_markers:
        if marker in subj_lower:
            issues.append(f"⚠ Possible objective data in Subjective: '{marker}'")
    
    for marker in subjective_markers:
        if marker in obj_lower:
            issues.append(f"⚠ Possible subjective data in Objective: '{marker}'")
    
    return issues

# Run validation
issues = validate_soap_separation(soap_data)

if issues:
    print("Cross-contamination warnings:")
    for issue in issues:
        print(f"  {issue}")
else:
    print("✓ No cross-contamination detected — Subjective and Objective are properly separated.")

## 3. Output: Formatted SOAP Note

In [ ]:
# ──────────────────────────────────────────────────────
# Format and display the SOAP note
# ──────────────────────────────────────────────────────

from IPython.display import Markdown, display

soap_markdown = (
    "# SOAP Note\n\n"
    f"## Subjective\n{soap_data['subjective']}\n\n"
    f"## Objective\n{soap_data['objective']}\n\n"
    f"## Assessment\n{soap_data['assessment']}\n\n"
    f"## Plan\n{soap_data['plan']}\n"
)

display(Markdown(soap_markdown))

In [ ]:
# ──────────────────────────────────────────────────────
# Save outputs
# ──────────────────────────────────────────────────────

# Save JSON
json_path = OUTPUT_DIR / "soap_note.json"
json_path.write_text(json.dumps(soap_data, indent=2), encoding="utf-8")
print(f"SOAP note (JSON) saved to:     {json_path}")

# Save Markdown
md_path = OUTPUT_DIR / "soap_note.md"
md_path.write_text(soap_markdown, encoding="utf-8")
print(f"SOAP note (Markdown) saved to: {md_path}")

print(f"\n{'─' * 60}")
print("All output files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name:25s}  ({f.stat().st_size:,} bytes)")

## 4. Scaling Considerations

If this pipeline were deployed at scale in a clinical environment:

| Concern | Approach |
|---------|----------|
| **Privacy/HIPAA** | Use faster-whisper locally — audio never leaves the facility. For SOAP generation, deploy a local LLM (Llama 3, Mistral) via Ollama. |
| **Throughput** | Batch process with faster-whisper on GPU (large-v3 model, float16). Process multiple files in parallel with async workers. |
| **Accuracy** | Fine-tune Whisper on medical dictation datasets (e.g., medical domain adaptation). Use larger Whisper models (medium/large-v3). |
| **Reliability** | Add retry logic with exponential backoff for API calls. Queue with Celery/Redis for async processing. |
| **Validation** | Expand cross-contamination checks with an NLI (Natural Language Inference) model. Flag uncertain sections for human review. |
| **Integration** | Output HL7 FHIR resources for EHR integration. Add speaker diarization for multi-speaker scenarios. |